# **SETUP**

In [1]:
import pathlib
import pandas as pd
from sklearn.preprocessing import KBinsDiscretizer
import warnings

warnings.filterwarnings(
    "ignore",
    message="Bins whose width are too small.*",
    category=UserWarning,
)

PROJECT_ROOT = pathlib.Path().absolute().parent

train = pd.read_parquet(PROJECT_ROOT / "data" / "train.parquet")
test = pd.read_parquet(PROJECT_ROOT / "data" / "test.parquet")
cv = pd.read_parquet(PROJECT_ROOT / "data" / "cv.parquet")

num_cols = train.select_dtypes("number").drop(columns=["PitStop", "PitNextLap"]).columns
n_bins = (
    train[num_cols]
    .nunique()
    .clip(lower=2, upper=10)
    .to_numpy()
)
kbins = KBinsDiscretizer(n_bins=n_bins, encode="onehot-dense", quantile_method='averaged_inverted_cdf').set_output(transform="pandas")

# **K-BINS DISCRETIZE NUMERICS**

In [2]:
train_subset = train[num_cols]
test_subset = test[num_cols]
oofs = []

for k in sorted(cv.outer_fold.unique()):
    is_val = cv["outer_fold"] == k
    kbins.fit(train_subset[~is_val])
    oofs.append(kbins.transform(train_subset[is_val]))

_ = kbins.fit(train_subset)

# **EXPORT**

In [3]:
feature_name = "005-kbins-discretize-numerics"
(PROJECT_ROOT / "data" / "features" / feature_name).mkdir(exist_ok=True)

oof_out = pd.concat(oofs).sort_index().astype("int8")
oof_out.to_parquet(PROJECT_ROOT / "data" / "features" / feature_name / "oof.parquet")

test_out = kbins.transform(test_subset).astype("int8")
test_out.to_parquet(PROJECT_ROOT / "data" / "features" / feature_name / "test.parquet")